<a href="https://colab.research.google.com/github/gph05010/Medication-Info-Text-Classification/blob/main/03_%EB%B2%A0%EC%9D%B4%EC%8A%A4%EB%9D%BC%EC%9D%B8_%EB%AA%A8%EB%8D%B8_%EC%BB%AC%EB%9F%BC_%EA%B8%B0%EC%A4%80_%EB%B6%84%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [ ]:
# sample_data = {
#     "text": [
#         "이 약은 속쓰림과 위산과다에 사용합니다.",
#         "성인은 1회 1정, 1일 3회 복용합니다.",
#         "임부는 복용 전 의사 또는 약사와 상의하십시오.",
#         "테트라사이클린계 항생제와 함께 복용하지 마십시오.",
#         "발진, 가려움 등이 나타나는 경우 복용을 중지하십시오.",
#         "습기와 빛을 피해 실온에서 보관하십시오.",
#         "이 약은 기침과 가래 증상 완화에 사용합니다.",
#         "어린이는 보호자의 지도하에 복용하십시오.",
#         "드물게 구역, 구토, 설사 등이 나타날 수 있습니다.",
#         "어린이의 손이 닿지 않는 곳에 보관하십시오."
#     ],
#     "label": [
#         "효능",
#         "복용법",
#         "주의사항",
#         "상호작용",
#         "부작용",
#         "보관법",
#         "효능",
#         "주의사항",
#         "부작용",
#         "보관법"
#     ]
# }

# df_model = pd.DataFrame(sample_data)

# df_model.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/해커톤

In [ ]:
# 컬럼 기준 분리 데이터셋
df_model = pd.read_csv("./medicine_column_label_dedup.csv")

df_model = df_model[["text", "label"]].dropna()
df_model["text"] = df_model["text"].astype(str)
df_model["label"] = df_model["label"].astype(str)

In [ ]:
# 문제 데이터, 정답 데이터, 학습 데이터, 테스트 데이터 분리
X = df_model["text"]
y = df_model["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train size:", len(X_train))
print("test size:", len(X_test))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

df_model = pd.read_csv("./medicine_column_label_dedup.csv")

df_model = df_model[["itemSeq", "itemName", "text", "label"]].dropna()
df_model["text"] = df_model["text"].astype(str)
df_model["label"] = df_model["label"].astype(str)
df_model["itemSeq"] = df_model["itemSeq"].astype(str)

X = df_model["text"]
y = df_model["label"]
groups = df_model["itemSeq"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("train size:", len(X_train))
print("test size:", len(X_test))

print("train itemSeq:", df_model.iloc[train_idx]["itemSeq"].nunique())
print("test itemSeq:", df_model.iloc[test_idx]["itemSeq"].nunique())

overlap = set(df_model.iloc[train_idx]["itemSeq"]) & set(df_model.iloc[test_idx]["itemSeq"])
print("train/test itemSeq overlap:", len(overlap))

In [ ]:
# 모델 학습 및 평가용 함수 정의
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    print("=" * 80)
    print(model_name)
    print("=" * 80)
    print("Accuracy:", accuracy)
    print("Macro F1:", f1_macro)
    print("Weighted F1:", f1_weighted)
    print()
    print(classification_report(y_test, y_pred, zero_division=0))

    result = {
        "model": model_name,
        "accuracy": accuracy,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted
    }

    return result, y_pred

In [ ]:
# TF-IDF + Logistic Regression
tfidf_logreg = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

result_tfidf_logreg, pred_tfidf_logreg = evaluate_model(
    "TF-IDF + Logistic Regression",
    tfidf_logreg,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# TF-IDF + LinearSVM
tfidf_svm = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", LinearSVC(
        random_state=42
    ))
])

result_tfidf_svm, pred_tfidf_svm = evaluate_model(
    "TF-IDF + LinearSVM",
    tfidf_svm,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# TF-IDF + Multinomial Naive Bayes
tfidf_nb = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", MultinomialNB())
])

result_tfidf_nb, pred_tfidf_nb = evaluate_model(
    "TF-IDF + Multinomial Naive Bayes",
    tfidf_nb,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
# CountVectorizer + Logistic Regression
count_logreg = Pipeline([
    ("vectorizer", CountVectorizer(
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

result_count_logreg, pred_count_logreg = evaluate_model(
    "CountVectorizer + Logistic Regression",
    count_logreg,
    X_train,
    X_test,
    y_train,
    y_test
)

In [ ]:
results = [
    result_tfidf_logreg,
    result_tfidf_svm,
    result_tfidf_nb,
    result_count_logreg
]

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(by="macro_f1", ascending=False)

result_df

In [ ]:
# 모델 성능 비교용 그래프
plot_df = result_df[["model", "accuracy", "macro_f1", "weighted_f1"]].set_index("model")

ax = plot_df.plot(kind="bar", figsize=(11, 5))

plt.title("Baseline Model Performance Comparison")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0.99, 1.00)
plt.xticks(rotation=30, ha="right")
plt.legend(loc="lower right")

# 막대 위에 값 표시
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
# Colab 한글 폰트 설치
!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)

plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 혼동행렬 그리기
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

labels = ["보관법", "복용법", "부작용", "상호작용", "주의사항", "효능"]

cm = confusion_matrix(y_test, pred_tfidf_svm, labels=labels)

# 행 기준 정규화: 실제 라벨별로 몇 %가 어디로 예측됐는지
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(9, 7))
im = plt.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

plt.title("Normalized Confusion Matrix - TF-IDF + LinearSVM", fontsize=15, pad=15)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)

plt.xticks(
    ticks=np.arange(len(labels)),
    labels=labels,
    rotation=45,
    ha="right",
    fontsize=11
)
plt.yticks(
    ticks=np.arange(len(labels)),
    labels=labels,
    fontsize=11
)

plt.colorbar(im, fraction=0.046, pad=0.04)

for i in range(len(labels)):
    for j in range(len(labels)):
        value = cm_norm[i, j]
        color = "white" if value > 0.5 else "black"
        plt.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color=color,
            fontsize=10
        )

plt.tight_layout()
plt.show()

In [ ]:
# 틀린 것 확인
error_df = pd.DataFrame({
    "text": X_test.values,
    "true_label": y_test.values,
    "pred_label": pred_tfidf_svm
})

error_df = error_df[error_df["true_label"] != error_df["pred_label"]]

error_df.head(20)

## 컬럼 기준 분리 데이터셋의 성능이 매우 높게 나온 이유

컬럼 기준 분리 데이터셋에서는 모델 성능이 매우 높게 측정되었다.  
처음에는 데이터 누수나 분할 방식의 문제를 의심했지만, `itemSeq` 기준으로 train/test를 분리한 뒤에도 오분류가 거의 발생하지 않았다.  
따라서 성능이 높게 나온 가장 큰 이유는 데이터셋 구성 방식 자체가 모델에게 비교적 쉬운 문제였기 때문으로 해석하였다.

### 1. 원본 데이터가 이미 항목별 컬럼으로 분리되어 있음

공공데이터포털의 의약품 개요정보 데이터는 처음부터 다음과 같이 항목별 컬럼으로 구조화되어 있었다.

| 원본 컬럼 | 라벨 |
|---|---|
| `efcyQesitm` | 효능 |
| `useMethodQesitm` | 복용법 |
| `atpnQesitm` | 주의사항 |
| `intrcQesitm` | 상호작용 |
| `seQesitm` | 부작용 |
| `depositMethodQesitm` | 보관법 |

즉, 컬럼 기준 데이터셋은 이미 항목별로 분리된 텍스트 덩어리를 다시 해당 라벨로 분류하는 구조이다.  
이 경우 모델은 사용자가 입력한 개별 문장을 분류하는 것이 아니라, 원본 데이터에서 이미 항목 단위로 정리된 긴 텍스트를 보고 라벨을 예측하게 된다.

따라서 컬럼 기준 데이터셋은 실제 사용 상황보다 문제 난이도가 낮다고 볼 수 있다.

### 2. 컬럼 단위 텍스트에는 라벨별 표현 단서가 많이 포함됨

컬럼 기준 데이터셋에서는 한 입력 안에 여러 문장이 함께 들어가기 때문에, 각 라벨을 구분할 수 있는 단서가 충분히 포함된다.

예를 들어 라벨별로 다음과 같은 표현이 반복적으로 등장한다.

| 라벨 | 자주 등장하는 표현 |
|---|---|
| 효능 | `이 약은 ~에 사용합니다`, `증상의 개선`, `완화` |
| 복용법 | `1회`, `1일`, `복용합니다`, `투여합니다` |
| 주의사항 | `복용하지 마십시오`, `상의하십시오`, `주의하십시오` |
| 상호작용 | `함께`, `병용`, `다른 약물`, `알코올` |
| 부작용 | `발진`, `가려움`, `구토`, `나타날 수 있습니다` |
| 보관법 | `보관하십시오`, `실온`, `습기`, `빛`, `어린이의 손` |

문장 단위 데이터셋에서는 하나의 문장만 보고 라벨을 판단해야 하지만, 컬럼 단위 데이터셋에서는 여러 문장이 함께 제공되므로 모델이 라벨별 단어 패턴을 훨씬 쉽게 학습할 수 있다.

### 3. 실제 서비스 상황과 차이가 있음

실제 서비스에서는 사용자가 종이 설명서나 약 봉투를 촬영하고, OCR을 통해 추출된 텍스트가 입력된다고 가정할 수 있다.  
이 경우 입력 텍스트는 효능, 복용법, 주의사항, 부작용 등이 명확히 분리된 컬럼 형태가 아니라, 여러 문장이 섞여 있는 형태에 가까울 가능성이 높다.

따라서 실제 모델이 해결해야 하는 문제는 다음에 가깝다.

> 추출된 설명문 안의 각 문장 또는 문단이 어떤 항목에 해당하는지 분류하는 것

반면 컬럼 기준 데이터셋은 이미 항목별 경계가 나뉜 상태이므로, 실제 사용 상황보다 더 쉬운 조건에서 평가된 결과라고 볼 수 있다.

### 4. 같은 의약품 중복 문제는 추가로 확인함

컬럼 기준 데이터셋에서는 같은 의약품의 여러 항목이 train과 test에 동시에 들어갈 가능성이 있었다.  
이를 확인하기 위해 `itemSeq` 기준으로 train/test를 분리하는 방식을 적용하였다.

그 결과 동일한 `itemSeq`가 train과 test에 동시에 포함되지 않도록 분리했음에도 성능이 매우 높게 나타났다.  
따라서 높은 성능의 주된 원인은 같은 의약품이 train/test에 섞였기 때문이라기보다, 컬럼 기준 텍스트 자체가 라벨별 특징을 강하게 포함하고 있기 때문으로 판단하였다.

### 5. 최종 해석

컬럼 기준 데이터셋에서 높은 성능이 나온 것은 모델이 잘못 평가되었다기보다는, 입력 데이터가 라벨을 구분하기 쉬운 형태였기 때문으로 해석된다.  
컬럼 단위 입력은 문맥이 충분히 유지되어 모델이 라벨별 특징을 잘 학습할 수 있지만, 실제 사용자가 입력하는 설명문 분류 상황과는 차이가 있다.

따라서 본 프로젝트에서는 컬럼 기준 데이터셋 결과를 최종 대표 성능으로 보기보다는, 문맥이 충분히 주어졌을 때의 참고 실험 결과로 활용하였다.  
실제 서비스 상황에 더 가까운 대표 실험은 문장/문단 단위 데이터셋을 기준으로 해석하는 것이 더 적절하다고 판단하였다.